In [ ]:
#%% --------------------------
# Install keras-tuner
#------------------------------
!pip install -q keras-tuner

#%% --------------------------
# Imports
#------------------------------
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelBinarizer
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
import kerastuner as kt

#%% --------------------------
# 1. Data Loading & Preprocessing
#------------------------------
df = pd.read_csv('OTDR_data.csv')

# Clean dataset
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

X = df[['SNR'] + [f'P{i}' for i in range(1, 31)]]
y_class = df['Class']

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y_class,
    test_size=0.3,
    stratify=y_class,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    stratify=y_temp,
    random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

lb = LabelBinarizer()
y_train = lb.fit_transform(y_train)
y_val = lb.transform(y_val)
y_test = lb.transform(y_test)

def prepare_inputs(X):
    return {
        'OTDR_trace': X[:, 1:],  # P1-P30
        'SNR': X[:, 0].reshape(-1, 1)  # SNR
    }

train_inputs = prepare_inputs(X_train)
val_inputs = prepare_inputs(X_val)
test_inputs = prepare_inputs(X_test)

#%% --------------------------
# 2. Model builder for Keras-Tuner
#------------------------------
def model_builder(hp):
    otdr_input = layers.Input(shape=(30,), name='OTDR_trace')
    snr_input = layers.Input(shape=(1,), name='SNR')

    x = layers.Reshape((30,1))(otdr_input)

    filters = hp.Choice('filters', values=[128, 256, 512])
    x = layers.Conv1D(filters, 3, activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Dropout(hp.Float('dropout_1', 0.3, 0.7, step=0.1))(x)

    lstm_units = hp.Choice('lstm_units', [64, 128, 256])
    x = layers.Bidirectional(layers.LSTM(lstm_units, return_sequences=True))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(hp.Float('dropout_2', 0.3, 0.7, step=0.1))(x)

    x = layers.Bidirectional(layers.LSTM(lstm_units//2))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(hp.Float('dropout_3', 0.3, 0.7, step=0.1))(x)

    snr_branch = layers.Dense(hp.Int('snr_dense_units', 32, 64, step=16), activation='relu')(snr_input)
    snr_branch = layers.BatchNormalization()(snr_branch)
    snr_branch = layers.Dropout(hp.Float('dropout_4', 0.3, 0.7, step=0.1))(snr_branch)

    combined = layers.Concatenate()([x, snr_branch])
    combined = layers.Dense(hp.Int('dense_units', 128, 256, step=64), activation='relu')(combined)
    combined = layers.BatchNormalization()(combined)
    combined = layers.Dropout(hp.Float('dropout_5', 0.3, 0.7, step=0.1))(combined)

    output = layers.Dense(8, activation='softmax')(combined)

    model = models.Model(inputs=[otdr_input, snr_input], outputs=output)

    lr = hp.Choice('learning_rate', [1e-3, 3e-4, 1e-4])
    model.compile(optimizer=optimizers.Adam(learning_rate=lr),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])

    return model

#%% --------------------------
# 3. Setup tuner and search
#------------------------------
tuner = kt.RandomSearch(
    model_builder,
    objective='val_accuracy',
    max_trials=20,
    executions_per_trial=1,
    directory='my_dir',
    project_name='otdr_tuning'
)

tuner.search(train_inputs, y_train,
             epochs=50,
             validation_data=(val_inputs, y_val),
             batch_size=64,
             callbacks=[tf.keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True)],
             verbose=2)

#%% --------------------------
# 4. Get best model and evaluate
#------------------------------
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
print("Best Hyperparameters:", best_hps.values)

model = tuner.hypermodel.build(best_hps)
history = model.fit(train_inputs, y_train,
                    validation_data=(val_inputs, y_val),
                    epochs=20,
                    batch_size=256,
                    callbacks=[tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)],
                    verbose=2)

test_loss, test_acc = model.evaluate(test_inputs, y_test, verbose=2)
print(f"Test accuracy: {test_acc:.4f}")

#%% --------------------------
# 5. Save model
#------------------------------
model.save('otdr_classification_model.h5')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.1/129.1 kB 2.4 MB/s eta 0:00:00


<ipython-input-1-491ed3f44b32>:15: DeprecationWarning: `import kerastuner` is deprecated, please use `import keras_tuner`.
  import kerastuner as kt



Search: Running Trial #1

Value             |Best Value So Far |Hyperparameter
512               |512               |filters
0.6               |0.6               |dropout_1
128               |128               |lstm_units
0.5               |0.5               |dropout_2
0.3               |0.3               |dropout_3
32                |32                |snr_dense_units
0.4               |0.4               |dropout_4
128               |128               |dense_units
0.4               |0.4               |dropout_5
0.0001            |0.0001            |learning_rate

Epoch 1/50


KeyboardInterrupt: 